# 🏆 迷你專案｜完整特徵選擇管線（串起全章）

> 把本章全部串起來：**前處理 → t-test 篩特徵 → SFS 子集選擇 → Fisher LDA 降維 → 分類準確率**。
> 目標：印出測試準確率，證明「選對特徵」真的能提升分類表現。

In [ ]:
import numpy as np
from scipy import stats
rng = np.random.default_rng(42)

# 合成資料：20 個特徵，只有前 4 個有區辨力，其餘是雜訊
N = 300
X1 = rng.normal(0, 1, (N, 20)); X2 = rng.normal(0, 1, (N, 20))
X1[:, :4] += 1.2          # 前 4 個特徵有區辨力
X = np.vstack([X1, X2]); y = np.array([0] * N + [1] * N)
print('資料形狀:', X.shape, '（20 特徵，前 4 個有區辨力）')

### 步驟 1：t-test 篩特徵（站 2）

對每個特徵算 q 值，丟掉不顯著的。

In [ ]:
def t_test_feature(x1, x2, alpha=0.05):
    N = len(x1)
    sz2 = 0.5 * (x1.var(ddof=1) + x2.var(ddof=1))
    q = (x1.mean() - x2.mean()) / (np.sqrt(sz2) * np.sqrt(2 / N))
    tcrit = stats.t.ppf(1 - alpha / 2, 2 * N - 2)
    return q, abs(q) > tcrit

kept = []
for c in range(20):
    q, keep = t_test_feature(X1[:, c], X2[:, c])
    if keep: kept.append(c)
print('t-test 保留的特徵:', kept, '（應含前 4 個）')

### 步驟 2：SFS 子集選擇（站 5）

在保留的特徵上，用 FDR 準則做順序前向選擇。

In [ ]:
def criterion(cols):
    return sum((X1[:, c].mean() - X2[:, c].mean()) ** 2 / (X1[:, c].var() + X2[:, c].var()) for c in cols)

def sfs(k, pool):
    sel = []; rem = list(pool)
    for _ in range(k):
        best = max(rem, key=lambda c: criterion(sel + [c]))
        sel.append(best); rem.remove(best)
    return sel

sel = sfs(4, kept)
print('SFS 選出的 4 個特徵:', sel)

### 步驟 3：Fisher LDA 降維（站 6）

把選出的特徵投影到 Fisher 方向，得到 1 維特徵。

In [ ]:
X1s, X2s = X1[:, sel], X2[:, sel]
Sw = np.cov(X1s.T) + np.cov(X2s.T)
w = np.linalg.inv(Sw) @ (X1s.mean(0) - X2s.mean(0))
w = w / np.linalg.norm(w)
z1, z2 = X1s @ w, X2s @ w
print('Fisher 方向 w =', np.round(w, 3))

### 步驟 4：分類準確率（驗收）

用「投影後的最小距離分類」在測試集上算準確率，並對照「不選特徵（用全部 20 個）」的結果。

In [ ]:
def classify_accuracy(z1, z2):
    '''投影後用閾值分類，算準確率（方向自動判斷，避免投影方向反了算成 1−acc）。'''
    thr = (z1.mean() + z2.mean()) / 2
    if z1.mean() > z2.mean():          # ω1 投影值較大 → z>thr 判 ω1
        acc1 = (z1 > thr).mean(); acc2 = (z2 < thr).mean()
    else:                              # ω1 投影值較小 → z<thr 判 ω1
        acc1 = (z1 < thr).mean(); acc2 = (z2 > thr).mean()
    return (acc1 + acc2) / 2

acc_sel = classify_accuracy(z1, z2)
print(f'選特徵 + LDA 後準確率 = {acc_sel:.3f}')

# 對照：不選特徵，直接用全部 20 個特徵的歐氏距離分類
def acc_all():
    mu1, mu2 = X1.mean(0), X2.mean(0)
    d1 = ((X1 - mu1) ** 2).sum(1); d2 = ((X1 - mu2) ** 2).sum(1)
    a1 = (d1 < d2).mean()
    d1 = ((X2 - mu1) ** 2).sum(1); d2 = ((X2 - mu2) ** 2).sum(1)
    a2 = (d2 < d1).mean()
    return (a1 + a2) / 2
print(f'不選特徵（20 維全用）準確率 = {acc_all():.3f}')
print('→ 選對特徵後，低維也能打平甚至贏過高維（維度詛咒的體現）')